In [1]:
import json

from openai import OpenAI
from minsearch import Index
from gitsource import GithubRepositoryDataReader, chunk_documents

openai_client = OpenAI()

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()
parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

def search(query):
    return index.search(query=query, num_results=5)

def llm_structured(user_prompt, instructions, output_format):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model="gpt-4o-mini",
        input=messages,
        text_format=output_format
    )

    return response.output_parsed

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 382 chunks from 95 documents


In [2]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question")
    found_answer: bool = Field(description="True if relevant information was found")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="Category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [3]:
def rag(query, output_format=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(prompt, instructions, output_format)

In [4]:
query = 'llm as a judge'

answer = rag(query)

print(f"found_answer: {answer.found_answer}")
print(f"confidence: {answer.confidence}")
print(f"answer_type: {answer.answer_type}")
print(f"answer: {answer.answer[:100]}...")

found_answer: True
confidence: 0.9
answer_type: explanation
answer: Using an LLM (Large Language Model) as a judge involves evaluating text responses based on custom cr...


uv add sentence-transformers par busqueda vectorial, esto instala pytorch
cargamos un modelo para embeddings
Usamos este modelo porque es pequeño y rapido optimizado para preguntas y respuestas

In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Data\formacion\ai-engineering-buildcamp\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\biarunxa\.cache\huggingface\hub\models--sentence-transformers--multi-qa-MiniLM-L6-cos-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ejemplo de como funciuonan los embeddings

In [6]:
q1 = 'install Evidently locally'
q2 = 'how can I set up evidently in my projects?'
v1 = model.encode(q1)
v1[:20]


array([ 0.07184878, -0.02879702,  0.06123494, -0.01810758,  0.12533097,
       -0.04134636, -0.06354985, -0.08986841, -0.04142815, -0.04465348,
        0.06421646,  0.0207709 ,  0.00219164,  0.00508831, -0.00833956,
       -0.0377002 ,  0.03059415, -0.0484462 ,  0.06848746, -0.01030118],
      dtype=float32)

In [7]:
v1.shape

(384,)

In [8]:
v2 = model.encode(q2)
v1.dot(v2)

np.float32(0.43477947)

In [9]:
q3 = 'how do i bake pretzels?'
v3 = model.encode(q3)
v1.dot(v3)

np.float32(0.022272602)

Creamos embeddings para todos nuestros documentos

In [10]:
texts = []

for doc in chunked_docs:
    title = doc.get('title', '')
    description = doc.get('description', '')
    content = doc.get('content', '')

    text = title + " " + description + " " + content
    texts.append(text.strip())

embeddings = model.encode(texts, show_progress_bar=True)
embeddings

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

array([[-0.0484977 ,  0.06601195,  0.01094064, ...,  0.03090534,
         0.06879062,  0.05070845],
       [-0.03232048, -0.01947779, -0.01677326, ...,  0.01519901,
         0.02379611, -0.00774647],
       [-0.02108491, -0.0565795 ,  0.00483409, ..., -0.04806893,
         0.0710432 ,  0.04317408],
       ...,
       [-0.00160592, -0.04139434, -0.05660459, ...,  0.05727456,
         0.09037897, -0.02732164],
       [-0.02243868, -0.00113358, -0.05355776, ...,  0.04080839,
         0.08891257, -0.00143775],
       [-0.06048303, -0.05174242,  0.01422585, ...,  0.00431389,
         0.09889208,  0.00393773]], shape=(382, 384), dtype=float32)

In [11]:
scores = embeddings.dot(v1)
scores[:15]

array([ 0.07959674,  0.12935524,  0.1866498 ,  0.26054287,  0.08233503,
       -0.02807788, -0.0721439 , -0.05210185, -0.01589226,  0.00971273,
       -0.05274091, -0.10643847,  0.04062286,  0.06344491, -0.04217618],
      dtype=float32)

In [12]:
from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields=['filename'])
vindex.fit(embeddings, chunked_docs)

In [13]:
def vector_search(query, num_results=5):
    vector = model.encode(query)
    return vindex.search(vector, num_results=num_results)

In [19]:
def rag_vector(query, output_format=RAGResponse):
    search_results = vector_search(query)
    print(search_results)
    prompt = build_prompt(query, search_results)
    return llm_structured(prompt, instructions, output_format)

In [20]:
answer = rag_vector(query)

print(f"Query: {query}")
print(f"found_answer: {answer.found_answer}")
print(f"confidence: {answer.confidence}")
print(f"answer: {answer.answer[:200]}...")

[{'start': 0, 'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create and tune the LLM evaluator**, which you can then apply in different contex

Aqui vamos a buscar la busqueda hybrida

In [16]:
def hybrid_search(query):
    vector_results = vector_search(query)
    text_results = search(query)
    return vector_results + text_results

In [17]:
def rag_hybrid(query, output_format=RAGResponse):
    search_results = hybrid_search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(prompt, instructions, output_format)

In [18]:
answer = rag_hybrid(query)

print(f"Query: {query}")
print(f"found_answer: {answer.found_answer}")
print(f"confidence: {answer.confidence}")
print(f"answer: {answer.answer[:200]}...")

Query: llm as a judge
found_answer: True
confidence: 0.95
answer: Using an LLM (Large Language Model) as a judge involves evaluating text based on custom criteria. This can be done in two ways: reference-based evaluation, which compares new responses against approve...
